# Clustering d'Images avec Scikit-learn et Visualisation avec Streamlit

Le clustering est une technique d'apprentissage non supervisé qui permet de regrouper des données similaires en différents groupes ou clusters. Dans le contexte de l'analyse d'images, le clustering peut être utilisé pour segmenter des images, détecter des objets ou des régions d'intérêt, et comprendre la structure des données d'image. Ce TP vous guidera à travers le processus de clustering d'un ensemble d'images de chiffres manuscrits en utilisant l'algorithme K-Means et l'extraction de caractéristiques HOG et d'histogrammes de niveaux de gris. Vous visualiserez ensuite les résultats du clustering à l'aide de la bibliothèque Streamlit.


À la fin de ce TP, vous aurez une compréhension pratique du processus de clustering d'images, de l'extraction de caractéristiques, de l'évaluation des performances et de la visualisation des résultats à l'aide de Streamlit. Ces compétences sont précieuses dans de nombreux domaines tels que la vision par ordinateur, le traitement d'images médicales, l'analyse de données d'imagerie satellitaire, et bien d'autres.


In [16]:
from sklearn.preprocessing import StandardScaler
import os
import pandas as pd
from sklearn import datasets

from features import *
from clustering import *
from utils import *
from constant import  PATH_OUTPUT, MODEL_CLUSTERING



## Partie 1 : Création du modèle de clustering d'images
#### (fichier pipeline.py)

**1. Chargement des données d'images de chiffres manuscrits à partir du dataset Digits.**
   - Vous utiliserez le célèbre dataset Digitis qui contient des images de chiffres manuscrits. Ce dataset est souvent utilisé pour tester des algorithmes de reconnaissance de chiffres et d'apprentissage automatique.



In [17]:
print("##### Chargement des données ######")
digits = datasets.load_digits()
labels_true =digits.target
images = digits.images


##### Chargement des données ######


**2. Extraction des caractéristiques HOG (Histogrammes de Gradients Orientés) et des histogrammes de niveaux de gris à partir des images.**
   - Les caractéristiques HOG capturent les informations de gradient et de bords dans les images, ce qui les rend utiles pour la reconnaissance d'objets et de formes.
   - Les histogrammes de niveaux de gris représentent la distribution des intensités de pixels dans l'image, fournissant des informations sur la texture et les motifs.

**TODO :**
   - Implémentez les fonctions `compute_hog_descriptors` et `compute_gray_histograms` dans  le fichier `features.py`, utilisez respectivement les fonctions `hog` de  la librairie `skimage` et  `calcHist` de `cv2`.
   - lien HOG : https://scikit-image.org/docs/stable/auto_examples/features_detection/plot_hog.html
   - lien  histogrammes de niveaux de gris : https://pyimagesearch.com/2021/04/28/opencv-image-histograms-cv2-calchist/
   

In [18]:

print("\n\n ##### Extraction de Features ######")
print("- calcul features hog...")
descriptors_hog = compute_hog_descriptors(images)
print("- calcul features Histogram...")
descriptors_hist = compute_gray_histograms(images)




 ##### Extraction de Features ######
- calcul features hog...
- calcul features Histogram...



**3. Application de l'algorithme K-Means sur les caractéristiques extraites pour obtenir les clusters.**
   - L'algorithme K-Means est un algorithme de clustering populaire qui partitionne les données en K clusters en minimisant la somme des carrés des distances entre les points de données et les centroïdes des clusters.
   
   
 
 **TODO :**
   - Dans le fichier `clustering.py` implémentez les fonctions `initialize_centers()`, `nearest_cluster()` et `fit()` du KMeans.
   

In [19]:
print("\n\n ##### Clustering ######")
number_cluster = 10  # Définition du nombre de clusters

# Initialisation de KMeans avec les descripteurs HOG et Histogram
kmeans_hog = KMeans(n_clusters=number_cluster)
kmeans_hist = KMeans(n_clusters=number_cluster)

print("- calcul kmeans avec features HOG ...")
kmeans_hog.fit(np.array(descriptors_hog))  # Entraînement du modèle KMeans avec les descripteurs HOG

print("- calcul kmeans avec features Histogram...")
kmeans_hist.fit(np.array(descriptors_hist))  # Entraînement du modèle KMeans avec les descripteurs Histogram




 ##### Clustering ######
- calcul kmeans avec features HOG ...
Centres initiaux: [[2.48502845e-01 2.48502845e-01 1.81732055e-01 7.22665430e-02
  5.22160292e-02 1.52848749e-01 7.05670300e-02 7.41169571e-02
  2.48502845e-01 2.48502845e-01 1.22641695e-01 1.08000215e-01
  1.72233104e-01 2.48502845e-01 1.13553000e-01 1.14994794e-01
  2.17364131e-01 0.00000000e+00 2.48502845e-01 1.17311605e-02
  8.06310820e-02 6.04093060e-02 1.60262480e-01 1.20611599e-01
  2.48502845e-01 2.34745285e-01 1.70711003e-01 2.48502845e-01
  2.31768260e-01 5.90451325e-02 1.70054369e-02 0.00000000e+00
  0.00000000e+00 8.51027441e-02 2.48502845e-01 2.00423591e-01]
 [2.85381653e-01 4.79345113e-02 1.46175327e-01 7.12975154e-02
  3.62503487e-02 5.59882771e-02 7.54961926e-02 5.24386981e-02
  2.85381653e-01 2.15907948e-01 0.00000000e+00 5.78090255e-03
  0.00000000e+00 7.76124652e-02 2.73893822e-01 1.59388026e-01
  4.06328643e-02 2.85381653e-01 1.72549972e-01 3.98320294e-02
  1.37348044e-01 2.56520188e-01 1.44587073e-01 2

**4. Évaluation des performances du clustering en utilisant des métriques telles que la pureté, l'entropie, et les scores Rand et Mutual Information.**
   - La pureté mesure la fraction d'exemples de cluster qui sont membres du cluster majoritaire.
   - L'entropie est une mesure de désordre ou d'impureté dans les clusters.
   - Le score Rand mesure la similarité entre deux partitions en comparant les paires d'exemples.
   - Le score Mutual Information évalue la quantité d'information partagée entre deux partitions.


In [20]:

print("\n\n##### Résultat ######")
metric_hist = show_metric(labels_true, kmeans_hist.labels_, descriptors_hist, bool_show=True, name_descriptor="HISTOGRAM", bool_return=True)
print("\n\n")
metric_hog = show_metric(labels_true, kmeans_hog.labels_, descriptors_hog,bool_show=True, name_descriptor="HOG", bool_return=True)




##### Résultat ######
########## Métrique descripteur : HISTOGRAM
Adjusted Rand Index: 0.03578347941471804
Jaccard Index: 0.05282510490587226
Homogeneity: 0.08005307557437719
Completeness: 0.08114122369645517
V-measure: 0.08059347684319916
Silhouette Score: 0.04527520388364792
Adjusted Mutual Information: 0.07132556524642611



########## Métrique descripteur : HOG
Adjusted Rand Index: 0.6307086143973617
Jaccard Index: 0.04443743399419807
Homogeneity: 0.7135133274557649
Completeness: 0.7164992661369072
V-measure: 0.7150031794007805
Silhouette Score: 0.13963714058104354
Adjusted Mutual Information: 0.7121455640807032


**5. Conversion des données de clustering au format requis pour la visualisation avec Streamlit.**


In [21]:
list_dict = [metric_hist,metric_hog]
df_metric = pd.DataFrame(list_dict)

scaler = StandardScaler()
descriptors_hist_norm = scaler.fit_transform(descriptors_hist)
descriptors_hog_norm = scaler.fit_transform(descriptors_hog)

#conversion vers un format 3D pour la visualisation
x_3d_hist = conversion_3d(descriptors_hist_norm)
x_3d_hog = conversion_3d(descriptors_hog_norm)

# création des dataframe pour la sauvegarde des données pour la visualisation
df_hist = create_df_to_export(x_3d_hist, labels_true, kmeans_hist.labels_)
df_hog = create_df_to_export(x_3d_hog, labels_true, kmeans_hog.labels_)

# Vérifie si le dossier existe déjà
if not os.path.exists(PATH_OUTPUT):
    # Crée le dossier
    os.makedirs(PATH_OUTPUT)

# sauvegarde des données
df_hist.to_excel(PATH_OUTPUT+"/save_clustering_hist_kmeans.xlsx")
df_hog.to_excel(PATH_OUTPUT+"/save_clustering_hog_kmeans.xlsx")
df_metric.to_excel(PATH_OUTPUT+"/save_metric.xlsx")




C:\Users\hadja\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(
C:\Users\hadja\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\manifold\_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


***6. Création du fichier pipeline.py*** 
- Mettez au propre le code dans le fichier pipeline.py
- Puis exécutez :  `python pipeline.py `


## Partie 2 : Visualisation des résultats du clustering avec Streamlit
### (fichier dashboad_clustering.py)

Cette partie constituera le rendu final du TP. Nous développerons une application Streamlit pour visualiser et analyser les résultats du clustering.

L'application permettra de :

***1. Visualisation 3D du clustering***
- Nous créerons une visualisation 3D interactive des clusters obtenus, avec la possibilité de mettre en évidence un cluster spécifique et d'afficher des exemples d'images appartenant à ce cluster.

***TODO :***
- Utilizer la fonction `scatter_3d()` pour faire un plot 3D du clustering.
- lien : https://plotly.com/python/3d-scatter-plots/




In [23]:
import matplotlib.pyplot as plt
import plotly.express as px


***2. Métriques d'évaluation***
- Nous calculerons et afficherons diverses métriques d'évaluation, telles que le score AMI (Adjusted Mutual Information), pour quantifier la qualité du clustering obtenu avec chaque descripteur.

***TODO :*** 
- Utilisez la fonction `px.bar()` pour afficher un histogramme du score AMI.
- lien : https://plotly.com/python/horizontal-bar-charts/

In [24]:
graph_size = 300


In [25]:
df_metric

,ami,ari,silhouette,homogeneity,completeness,v_measure,jaccard,descriptor,name_model
0,0.071326,0.035783,0.045275,0.080053,0.081141,0.080593,0.052825,HISTOGRAM,kmeans
1,0.712146,0.630709,0.139637,0.713513,0.716499,0.715003,0.044437,HOG,kmeans


***3. Finalisation du fichier dashboard_clustering.py***

***TODO :***
- Ajoutez les graphiques dans le fichier, puis lancez la commande :  `streamlit run dashboard_clustering.py `